# MCDI505 · Semana 4 — Evaluación final en 2024 (anexo del informe)
**Estudiante:** _Nombre Apellido_ · **Archivo:** `mcdi505_s4inicialapellido.ipynb`

Este notebook **no se califica por separado**: es el anexo que respalda las cifras de su informe ejecutivo. Complete la especificación de su modelo final (Sumativa 3), ejecute todo y lleve al informe la tabla y la figura que produce.

## 0. Configuración y datos

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
warnings.filterwarnings("ignore")
SEMILLA = 505
np.random.seed(SEMILLA)

In [ ]:
# ==============================================================================
# Funciones de apoyo (las mismas de la Sesión 3, más tres envoltorios para DEL)
# No necesita modificarlas.
# ==============================================================================

def calcular_exponente_hurst(serie, min_ventana=8, max_ventana=None, n_ventanas=10):
    """Exponente de Hurst por Rango Reescalado (R/S). Ver Sesión 3, §2.3."""
    serie = np.asarray(serie, dtype=float)
    n = len(serie)
    if max_ventana is None:
        max_ventana = n // 4
    ventanas = np.unique(np.geomspace(min_ventana, max_ventana, num=n_ventanas, dtype=int))
    rs_promedio, ventanas_validas = [], []
    for w in ventanas:
        m = n // w
        if m < 1:
            continue
        rs_bloques = []
        for i in range(m):
            bloque = serie[i * w:(i + 1) * w]
            recorrido = np.cumsum(bloque - np.mean(bloque))
            desviacion = np.std(bloque)
            if desviacion > 0:
                rs_bloques.append((np.max(recorrido) - np.min(recorrido)) / desviacion)
        if rs_bloques:
            rs_promedio.append(np.mean(rs_bloques))
            ventanas_validas.append(w)
    pendiente, _ = np.polyfit(np.log(ventanas_validas), np.log(rs_promedio), 1)
    return pendiente


def aplicar_filtro_fraccional(serie, d):
    """Operador (1-B)^d. d > 0 diferencia fraccionalmente; d < 0 integra. Ver Sesión 3, §2.3."""
    serie = np.asarray(serie, dtype=float)
    n = len(serie)
    pesos = np.zeros(n)
    pesos[0] = 1.0
    for k in range(1, n):
        pesos[k] = pesos[k - 1] * (k - 1 - d) / k
    return np.array([np.sum(pesos[:t + 1] * serie[t::-1]) for t in range(n)])


def garch_log_verosimilitud_negativa(parametros, serie):
    """Log-verosimilitud negativa de un GARCH(1,1) gaussiano. Ver Sesión 3, §3.4."""
    omega, alpha, beta = parametros
    sigma2 = np.zeros(len(serie))
    sigma2[0] = np.var(serie)
    for t in range(1, len(serie)):
        sigma2[t] = omega + alpha * serie[t - 1] ** 2 + beta * sigma2[t - 1]
    if np.any(sigma2 <= 1e-8):
        return 1e10
    return 0.5 * np.sum(np.log(2 * np.pi) + np.log(sigma2) + serie ** 2 / sigma2)


def correccion_arfima(residuos, d, h):
    """Pronóstico h pasos del componente ARFIMA(0,d,0) de los residuos.

    Diferencia fraccionalmente los residuos con d, pronostica cero (ruido blanco)
    y vuelve a integrar con -d. Lo que queda es la persistencia que la memoria
    larga añadiría al pronóstico del SARIMA (en escala logarítmica).
    """
    r = np.asarray(residuos, dtype=float)
    extendida = np.concatenate([aplicar_filtro_fraccional(r, d), np.zeros(h)])
    return aplicar_filtro_fraccional(extendida, -d)[len(r):]


def estimar_garch11(residuos, h):
    """Estima un GARCH(1,1) por MLE sobre los residuos estandarizados.

    Devuelve un diccionario con omega, alpha, beta, persistencia, convergencia,
    residuos estandarizados por sigma_t y el factor de escala de la desviación
    estándar condicional para los h meses siguientes (1 = igual a la varianza
    incondicional).
    """
    r = np.asarray(residuos, dtype=float)
    s = r.std()
    u = r / s
    res = minimize(garch_log_verosimilitud_negativa, x0=[0.05, 0.10, 0.80], args=(u,),
                   bounds=[(1e-6, 5.0), (1e-6, 0.999), (1e-6, 0.999)],
                   constraints=({"type": "ineq", "fun": lambda p: 0.999 - p[1] - p[2]},),
                   method="SLSQP")
    om, al, be = res.x
    sigma2 = np.zeros(len(u)); sigma2[0] = u.var()
    for t in range(1, len(u)):
        sigma2[t] = om + al * u[t - 1] ** 2 + be * sigma2[t - 1]
    futura = np.zeros(h)
    futura[0] = om + al * u[-1] ** 2 + be * sigma2[-1]
    for i in range(1, h):
        futura[i] = om + (al + be) * futura[i - 1]
    incondicional = om / (1 - al - be)
    return {"omega": om, "alpha": al, "beta": be, "persistencia": al + be,
            "convergencia": bool(res.success),
            "residuos_estandarizados": u / np.sqrt(sigma2),
            "escala_futura": np.sqrt(futura / incondicional)}


def metricas(real, pred):
    """RMSE y MAE en la escala de los datos (GWh)."""
    e = np.asarray(real, float) - np.asarray(pred, float)
    return {"RMSE": float(np.sqrt(np.mean(e ** 2))), "MAE": float(np.mean(np.abs(e)))}


def cobertura_y_ancho(real, inferior, superior):
    """Proporción de meses dentro del intervalo y ancho promedio del intervalo (GWh)."""
    real, inferior, superior = (np.asarray(x, float) for x in (real, inferior, superior))
    return {"Cobertura": float(np.mean((real >= inferior) & (real <= superior))),
            "Ancho medio": float(np.mean(superior - inferior))}

In [ ]:
df = pd.read_csv("consumo_electrico_litoral_depurado.csv", parse_dates=["fecha"], index_col="fecha")
df.index.freq = "MS"
y = df["consumo_gwh"]
log_y = np.log(y)

In [ ]:
# ==============================================================================
# COMPLETE: el modelo SARIMA que seleccionó en la Sumativa 2 (misma especificación)
# ==============================================================================
ORDEN = None              # por ejemplo (p, d, q)
ORDEN_ESTACIONAL = None   # por ejemplo (P, D, Q, 12)
USAR_COVID = False        # True si incluyó la indicadora covid como variable exógena

assert ORDEN is not None and ORDEN_ESTACIONAL is not None, "Complete ORDEN y ORDEN_ESTACIONAL con su modelo de la Sumativa 2."

# COMPLETE solo si en la Sumativa 3 su modelo final incorporó componentes avanzados
D_FRACCIONAL = None       # valor de d si su modelo final incluye la corrección ARFIMA(0,d,0); None si no
BANDAS_GARCH = False      # True si su modelo final usa bandas de volatilidad GARCH(1,1)

In [ ]:
def ajustar_y_pronosticar(fin_entrenamiento, inicio_prueba, fin_prueba):
    """Ajusta el modelo final hasta `fin_entrenamiento` y pronostica el periodo de prueba en GWh."""
    lt = log_y[:fin_entrenamiento]
    h = len(log_y[inicio_prueba:fin_prueba])
    Xt = df[["covid"]][:fin_entrenamiento] if USAR_COVID else None
    Xp = df[["covid"]].loc[inicio_prueba:fin_prueba] if USAR_COVID else None
    m = SARIMAX(lt, exog=Xt, order=ORDEN, seasonal_order=ORDEN_ESTACIONAL).fit(disp=False)
    inicio = ORDEN[1] + ORDEN_ESTACIONAL[1] * ORDEN_ESTACIONAL[3]
    resid = m.resid.iloc[inicio:]
    p = m.get_forecast(h, exog=Xp)
    media = p.predicted_mean.to_numpy()
    sd = ((p.conf_int(alpha=0.05).iloc[:, 1] - p.predicted_mean) / 1.96).to_numpy()
    if D_FRACCIONAL is not None:
        media = media + correccion_arfima(resid, D_FRACCIONAL, h)
    if BANDAS_GARCH:
        sd = sd * estimar_garch11(resid, h)["escala_futura"]
    return np.exp(media), np.exp(media - 1.96 * sd), np.exp(media + 1.96 * sd)


def evaluar(fin_entrenamiento, inicio_prueba, fin_prueba):
    """Métricas del modelo final y del naive estacional sobre el periodo de prueba."""
    real = y[inicio_prueba:fin_prueba].to_numpy()
    pred, inf, sup = ajustar_y_pronosticar(fin_entrenamiento, inicio_prueba, fin_prueba)
    naive = y.shift(12)[inicio_prueba:fin_prueba].to_numpy()
    y_ent = y[:fin_entrenamiento]
    escala_mase = float((y_ent - y_ent.shift(12)).dropna().abs().mean())   # MAE del naive estacional en entrenamiento
    filas = {}
    for nombre, p in [("Modelo final", pred), ("Naive estacional", naive)]:
        m = metricas(real, p)
        m["MASE"] = float(np.mean(np.abs(real - p)) / escala_mase)
        filas[nombre] = m
    tabla = pd.DataFrame(filas).T
    tabla["Cobertura IC 95 %"] = [cobertura_y_ancho(real, inf, sup)["Cobertura"], np.nan]
    return tabla, pred, inf, sup

## 1. Evaluación en validación 2023 y en prueba 2024

In [ ]:
tabla_2023, *_ = evaluar("2022-12", "2023-01", "2023-12")
tabla_2024, pred24, inf24, sup24 = evaluar("2023-12", "2024-01", "2024-12")

resumen = pd.concat({"Validación 2023": tabla_2023, "Prueba 2024": tabla_2024})
print(resumen.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y["2021-01":"2024-12"].index, y["2021-01":"2024-12"], color="#333333", lw=1.4, label="consumo observado")
ax.plot(y["2024"].index, pred24, color="#b30338", lw=1.8, ls="--", label="pronóstico 2024 (modelo final)")
ax.fill_between(y["2024"].index, inf24, sup24, color="#b30338", alpha=0.15, label="intervalo al 95 %")
ax.axvline(pd.Timestamp("2024-01-01"), color="#777777", ls=":", lw=1)
ax.set_title("Consumo eléctrico comercial de DEL: pronóstico 2024 frente a lo observado", loc="left")
ax.set_ylabel("GWh"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

## 2. Notas para el informe

- **Tabla:** lleve al informe `resumen`, reformateada como tabla del documento (no como salida de consola), junto con las cifras de validación de sus semanas 2 y 3.
- **Figura:** la figura anterior es la figura clave del informe.
- **MASE < 1** indica que el modelo supera al naive estacional; **MASE > 1**, que no lo supera. Interprete el resultado, sea cual sea.
- **Cobertura:** proporción de meses de la prueba que caen dentro del intervalo al 95 %.

## Declaración de uso de asistentes de IA

| Pregunta | Respuesta |
|---|---|
| ¿Utilizó asistentes de IA en esta entrega? | |
| ¿En qué partes específicas? | |
| ¿Qué verificó de forma independiente y cómo? | |